# DeepFashion-MultiModal 기반 현재 모델 평가

공식 데이터의 소매·하의 길이·소재·패턴·네크라인 정답과 현재 파이프라인 출력을 비교합니다. 데이터는 비상업 연구용이며 이 프로젝트에 포함하거나 재배포하지 않습니다.

## 1. 팀원별 데이터 경로 설정

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_DIR_INPUT = r''  # 비우면 현재 폴더에서 자동 탐색
DEEPFASHION_ROOT_INPUT = r''  # 공식 DeepFashion-MultiModal 폴더
IMAGE_DIR_INPUT = r'image'
SHAPE_LABEL_INPUT = r'labels/shape_anno_all.txt'
FABRIC_LABEL_INPUT = r'labels/fabric_anno_all.txt'
PATTERN_LABEL_INPUT = r'labels/color_anno_all.txt'
MAX_SAMPLES = 20  # CPU에서는 먼저 20장 정도로 확인

if PROJECT_DIR_INPUT:
    PROJECT_DIR = Path(PROJECT_DIR_INPUT).expanduser().resolve()
else:
    candidates = [Path.cwd(), Path.cwd() / 'ai_fashion_recommender']
    PROJECT_DIR = next((p.resolve() for p in candidates if (p / 'config.py').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('PROJECT_DIR_INPUT에 프로젝트 폴더를 입력하세요.')
if not DEEPFASHION_ROOT_INPUT:
    raise ValueError('DEEPFASHION_ROOT_INPUT에 공식 데이터 폴더를 입력하세요.')
DEEPFASHION_ROOT = Path(DEEPFASHION_ROOT_INPUT).expanduser().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))

## 2. 공식 라벨 결합

In [ ]:
from clothing_parser import ClothingParser
from deepfashion_dataset import load_deepfashion_multimodal, evaluate_deepfashion_predictions
from fashion_model import FashionClassifier
from outfit_analyzer import OutfitAnalyzer
from pose_analyzer import PoseAnalyzer

records = load_deepfashion_multimodal(
    DEEPFASHION_ROOT / IMAGE_DIR_INPUT,
    DEEPFASHION_ROOT / SHAPE_LABEL_INPUT,
    DEEPFASHION_ROOT / FABRIC_LABEL_INPUT,
    DEEPFASHION_ROOT / PATTERN_LABEL_INPUT,
)
print(f'평가 가능한 이미지: {len(records):,}장')

## 3. 현재 모델로 예측하고 정확도 계산

In [ ]:
pose_analyzer = PoseAnalyzer(model_complexity=1)
outfit_analyzer = OutfitAnalyzer(
    ClothingParser(use_fashn=True),
    FashionClassifier(enabled=True),
)

def predict(image_path):
    pose = pose_analyzer.analyze(image_path)
    if not pose.valid:
        return {}
    result, _ = outfit_analyzer.analyze(image_path, pose)
    return result.to_dict()

report = evaluate_deepfashion_predictions(records, predict, max_samples=MAX_SAMPLES)
print(json.dumps(report['metrics'], ensure_ascii=False, indent=2))

## 4. 오답 사례와 보고서 저장

In [ ]:
report_path = PROJECT_DIR / 'outputs' / 'deepfashion_evaluation.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('대표 오답:', json.dumps(report['mismatches'][:10], ensure_ascii=False, indent=2))
print('저장 위치:', report_path)
pose_analyzer.close()